# Assistente de voz com Whisper e ChatGPT

Execute no **Google Colab**, da primeira à última célula. O navegador grava cinco segundos; o Whisper `small` transcreve no Colab, o GPT-4 responde pela API e o gTTS gera a voz em português.

**Dados e custos:** a pergunta transcrita vai para a OpenAI; a resposta vai ao serviço do Google usado pelo gTTS. A API da OpenAI pode gerar cobrança separada da assinatura ChatGPT. Os áudios ficam na sessão e podem ser incorporados ao notebook ao salvar saídas. Limpe as saídas antes de compartilhar.

Configure `OPENAI_API_KEY` em **Segredos** (ícone de chave do Colab), habilitando o acesso deste notebook, ou na variável de ambiente da sessão. Não escreva nem imprima a chave em uma célula. A chave só será necessária na etapa OpenAI.

## 1. Configuração e preparação

Uma sessão nova reduz conflitos com pacotes de outros projetos. CPU funciona; uma GPU pode acelerar o Whisper. A primeira transcrição baixa o modelo. Cada pergunta é independente, sem histórico de conversa.

In [ ]:
SETTINGS = {
    "language": "pt",
    "record_seconds": 5,
    "whisper_model": "small",
    "openai_model": "gpt-4",
}

# Executar esta célula reinicia o estado da pergunta, sem alterar sua chave.
record_file = transcription = chatgpt_response = response_audio = None
_assistant_ready = False
_preparation_complete = False

In [ ]:
_preparation_complete = False
_assistant_ready = False

import shutil
import subprocess
import sys

try:
    from google.colab import output
except ImportError:
    raise RuntimeError("Abra este notebook no Google Colab para gravar pelo navegador.") from None

if shutil.which("ffmpeg") is None:
    raise RuntimeError(
        "FFmpeg não foi encontrado. Use um ambiente Colab com FFmpeg instalado "
        "e execute novamente a preparação antes de gravar."
    )

# Usa o Python do kernel; uma instalação que falhar interrompe a célula.
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "openai==2.54.0", "openai-whisper==20250625", "gTTS==2.5.4",
], check=True)

dependency_check = subprocess.run(
    [sys.executable, "-m", "pip", "check"], capture_output=True, text=True
)
if dependency_check.returncode:
    print("Aviso: há conflitos com pacotes deste ambiente:")
    print(dependency_check.stdout)
    print("Confira o README. Não atualize pacotes de outras ferramentas indiscriminadamente.")
else:
    print("Dependências verificadas. Continue com a célula de funções abaixo.")
_preparation_complete = True

### Funções do assistente

Execute esta célula uma vez após a preparação. Ela define as funções; a gravação e os serviços externos só são chamados nas etapas seguintes.

In [ ]:
import base64
import binascii
import json
import math
import os
from pathlib import Path
import shutil
import tempfile

from IPython.display import Audio, Javascript, display

RECORD = r"""
async function recordAudio(durationMs) {
  if (!Number.isFinite(durationMs) || durationMs <= 0) {
    throw new Error("A duração da gravação deve ser positiva e finita.");
  }
  if (!navigator.mediaDevices?.getUserMedia || typeof MediaRecorder === "undefined") {
    throw new Error("Este navegador não oferece gravação de áudio. Use um navegador compatível no Colab por HTTPS.");
  }
  let stream;
  let recorder;
  try {
    stream = await navigator.mediaDevices.getUserMedia({audio: true});
    const types = ["audio/webm;codecs=opus", "audio/webm", "audio/ogg;codecs=opus", "audio/mp4"];
    const supported = types.find(type => MediaRecorder.isTypeSupported(type));
    recorder = supported ? new MediaRecorder(stream, {mimeType: supported}) : new MediaRecorder(stream);
    const chunks = [];
    const blob = await new Promise((resolve, reject) => {
      let timer;
      recorder.ondataavailable = event => {
        if (event.data?.size > 0) chunks.push(event.data);
      };
      recorder.onerror = () => {
        clearTimeout(timer);
        reject(new Error("O navegador não conseguiu concluir a gravação. Tente novamente."));
      };
      recorder.onstop = () => {
        clearTimeout(timer);
        const mimeType = recorder.mimeType || chunks[0]?.type;
        const audio = new Blob(chunks, {type: mimeType || ""});
        if (!audio.size) reject(new Error("A gravação ficou vazia. Confira o microfone e tente novamente."));
        else resolve(audio);
      };
      recorder.start();
      timer = setTimeout(() => {
        try {
          if (recorder.state !== "inactive") recorder.stop();
        } catch (error) {
          reject(new Error("Não foi possível encerrar a gravação."));
        }
      }, durationMs);
    });
    const dataUrl = await new Promise((resolve, reject) => {
      const reader = new FileReader();
      reader.onload = () => resolve(reader.result);
      reader.onerror = () => reject(new Error("Não foi possível ler o áudio gravado."));
      reader.onabort = () => reject(new Error("A leitura do áudio foi interrompida."));
      reader.readAsDataURL(blob);
    });
    return {dataUrl, mimeType: blob.type};
  } catch (error) {
    if (["NotAllowedError", "SecurityError"].includes(error.name)) {
      throw new Error("Permissão do microfone negada. Autorize o microfone no navegador e execute a gravação novamente.");
    }
    if (error.name === "NotFoundError") {
      throw new Error("Nenhum microfone foi encontrado. Conecte um microfone e tente novamente.");
    }
    throw error;
  } finally {
    try {
      if (recorder && recorder.state !== "inactive") recorder.stop();
    } catch (_) {}
    if (stream) {
      for (const track of stream.getTracks()) {
        try { track.stop(); } catch (_) {}
      }
    }
  }
}
"""

def record(sec=5):
    """Grava no navegador Colab e retorna o caminho do arquivo no formato real."""
    if isinstance(sec, bool) or not isinstance(sec, (int, float)) or not math.isfinite(sec) or sec <= 0:
        raise ValueError("A duração da gravação deve ser um número positivo e finito.")
    from google.colab import output
    display(Javascript(RECORD))
    try:
        captured = output.eval_js(f"recordAudio({json.dumps(sec * 1000)})")
    except Exception:
        raise RuntimeError(
            "Não foi possível gravar. Confira a permissão do microfone, o dispositivo "
            "e a conexão com o Colab; depois execute novamente a gravação."
        ) from None
    if not isinstance(captured, dict):
        raise RuntimeError("O navegador retornou uma gravação inválida. Grave novamente.")
    data_url = captured.get("dataUrl", "")
    if not isinstance(data_url, str) or "," not in data_url:
        raise RuntimeError("O navegador não retornou o áudio esperado. Grave novamente.")
    header, encoded = data_url.split(",", 1)
    mime = str(captured.get("mimeType", "")).split(";", 1)[0].lower()
    header_mime = header.removeprefix("data:").split(";", 1)[0].lower()
    extensions = {
        "audio/webm": ".webm", "video/webm": ".webm",
        "audio/ogg": ".ogg", "application/ogg": ".ogg",
        "audio/mp4": ".m4a", "video/mp4": ".m4a",
        "audio/wav": ".wav", "audio/x-wav": ".wav", "audio/mpeg": ".mp3",
    }
    if not header.startswith("data:") or not header.endswith(";base64") or mime not in extensions or header_mime != mime:
        raise RuntimeError("O formato de áudio retornado pelo navegador não é compatível.")
    try:
        audio = base64.b64decode(encoded, validate=True)
    except (binascii.Error, ValueError):
        raise RuntimeError("O áudio recebido está corrompido. Grave novamente.") from None
    if not audio:
        raise RuntimeError("A gravação ficou vazia. Confira o microfone e tente novamente.")
    path = Path("request_audio" + extensions[mime]).resolve()
    try:
        path.write_bytes(audio)
    except OSError:
        raise RuntimeError("Não foi possível salvar o áudio na sessão. Confira o espaço disponível.") from None
    return str(path)


def transcribe_audio(audio_path, language="pt", model_name="small"):
    """Executa Whisper localmente no ambiente Colab, sem API de transcrição."""
    if not audio_path:
        raise RuntimeError("Execute a etapa de gravação antes da transcrição.")
    path = Path(audio_path)
    if not path.is_file() or not path.stat().st_size:
        raise RuntimeError("O arquivo de áudio não existe ou está vazio. Grave novamente.")
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("FFmpeg não foi encontrado. Execute a preparação antes da transcrição.")
    import whisper
    try:
        model = whisper.load_model(model_name)
        result = model.transcribe(str(path), fp16=False, language=language)
    except Exception:
        raise RuntimeError(
            "Falha na transcrição. Confira o áudio, o FFmpeg, a memória e a conexão "
            "para baixar o modelo Whisper."
        ) from None
    text = result.get("text") if isinstance(result, dict) else None
    if not isinstance(text, str) or not text.strip():
        raise RuntimeError("Nenhuma fala foi reconhecida. Grave uma nova pergunta antes de continuar.")
    return text.strip()


def read_api_key():
    """Lê a credencial sem modificar o ambiente nem imprimir o segredo."""
    key = os.environ.get("OPENAI_API_KEY", "").strip()
    if not key:
        try:
            from google.colab import userdata
            key = userdata.get("OPENAI_API_KEY")
        except Exception:
            raise RuntimeError(
                "Configure OPENAI_API_KEY nos Segredos do Colab e autorize o acesso "
                "deste notebook, ou configure a variável de ambiente. Não cole a chave no código."
            ) from None
    if not isinstance(key, str) or not key.strip() or key.strip().upper() in {
        "API KEY AQUI", "SUA_CHAVE_AQUI", "YOUR_API_KEY", "OPENAI_API_KEY", "SK-EXAMPLE", "SK-PLACEHOLDER",
    }:
        raise RuntimeError("OPENAI_API_KEY está ausente ou contém um texto de exemplo. Configure uma chave válida com segurança.")
    return key.strip()


def ask_chatgpt(text, model_name="gpt-4"):
    """Envia uma pergunta independente usando o SDK atual e retorna texto completo."""
    if not isinstance(text, str) or not text.strip():
        raise RuntimeError("Não há uma transcrição válida. Execute a gravação e a transcrição primeiro.")
    from openai import (
        OpenAI, AuthenticationError, PermissionDeniedError, RateLimitError,
        APITimeoutError, APIConnectionError, APIStatusError, OpenAIError,
    )
    key = read_api_key()
    try:
        with OpenAI(api_key=key, timeout=60.0) as client:
            response = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": text.strip()}],
            )
    except AuthenticationError:
        raise RuntimeError("A OpenAI recusou a autenticação. Confira a chave configurada nos Segredos ou no ambiente.") from None
    except PermissionDeniedError:
        raise RuntimeError("A conta não tem permissão para esta solicitação. Confira o projeto e o acesso ao modelo na OpenAI.") from None
    except RateLimitError:
        raise RuntimeError("A OpenAI informou limite de uso ou cota insuficiente. Confira faturamento e limites antes de tentar novamente.") from None
    except APITimeoutError:
        raise RuntimeError("A OpenAI demorou a responder. Tente novamente mais tarde.") from None
    except APIConnectionError:
        raise RuntimeError("Não foi possível conectar à OpenAI. Confira a conexão e tente novamente.") from None
    except APIStatusError:
        raise RuntimeError("A OpenAI recusou a solicitação. Confira a disponibilidade do modelo e tente novamente mais tarde.") from None
    except OpenAIError:
        raise RuntimeError("Não foi possível obter a resposta da OpenAI. Confira a configuração e tente novamente.") from None
    if not response.choices:
        raise RuntimeError("A OpenAI não retornou uma resposta. Tente novamente.")
    choice = response.choices[0]
    if choice.finish_reason != "stop":
        raise RuntimeError("A resposta não foi concluída normalmente. Tente novamente antes de gerar a voz.")
    answer = choice.message.content
    if not isinstance(answer, str) or not answer.strip():
        raise RuntimeError("A OpenAI não retornou texto para sintetizar. Tente novamente.")
    return answer.strip()


def synthesize_speech(text, language="pt", destination="response_audio.mp3"):
    """Gera MP3 com gTTS; só substitui o arquivo final após uma escrita completa."""
    if not isinstance(text, str) or not text.strip():
        raise RuntimeError("Não há uma resposta válida para gerar voz. Execute a etapa OpenAI primeiro.")
    path = Path(destination).resolve()
    if path.suffix.lower() != ".mp3":
        raise ValueError("O arquivo de saída do gTTS deve ter extensão .mp3.")
    from gtts import gTTS, gTTSError
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(dir=path.parent, prefix=".response-", suffix=".mp3", delete=False) as handle:
            temporary = Path(handle.name)
        speech = gTTS(text=text.strip(), lang=language, slow=False, timeout=(10, 60))
        speech.save(str(temporary))
        if not temporary.stat().st_size:
            raise RuntimeError("O serviço de voz retornou um áudio vazio. Tente novamente.")
        temporary.replace(path)
    except gTTSError:
        raise RuntimeError("Não foi possível gerar a voz pelo Google. Confira a conexão e tente novamente.") from None
    except ValueError:
        raise RuntimeError("Não foi possível gerar voz com o idioma configurado. Confira a configuração.") from None
    except OSError:
        raise RuntimeError("Não foi possível salvar a resposta em áudio. Confira o espaço e as permissões da sessão.") from None
    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)
    return str(path)


_assistant_ready = True
print("Funções carregadas. Continue com a gravação.")

## 2. Gravar a pergunta

Autorize o microfone quando solicitado e fale durante cinco segundos. Se você negar a permissão, ajuste-a no navegador e execute esta célula novamente. Uma nova tentativa invalida a transcrição e a resposta anteriores.

In [ ]:
record_file = transcription = chatgpt_response = response_audio = None
if not globals().get("_assistant_ready", False) or not globals().get("_preparation_complete", False) or "SETTINGS" not in globals():
    raise RuntimeError("Execute as células de configuração, preparação e funções, nesta ordem, antes de continuar.")

print("Ouvindo...")
record_file = record(SETTINGS["record_seconds"])
display(Audio(filename=record_file, autoplay=False))

## 3. Transcrever com Whisper

O modelo é executado no Colab. O primeiro carregamento pode demorar e exige internet para baixar os pesos. Confira a transcrição antes de enviar a pergunta à OpenAI.

In [ ]:
transcription = chatgpt_response = response_audio = None
if not globals().get("_assistant_ready", False) or not globals().get("_preparation_complete", False) or "SETTINGS" not in globals():
    raise RuntimeError("Execute as células de configuração, preparação e funções, nesta ordem, antes de continuar.")

transcription = transcribe_audio(
    globals().get("record_file"), SETTINGS["language"], SETTINGS["whisper_model"]
)
print(transcription)

## 4. Obter a resposta da OpenAI

**Esta célula faz uma chamada de API e pode gerar cobrança.** Configure a chave nos Segredos do Colab ou no ambiente antes de executar. O modelo permanece `gpt-4`; a disponibilidade depende da sua conta. Não existe memória de perguntas anteriores.

In [ ]:
chatgpt_response = response_audio = None
if not globals().get("_assistant_ready", False) or not globals().get("_preparation_complete", False) or "SETTINGS" not in globals():
    raise RuntimeError("Execute as células de configuração, preparação e funções, nesta ordem, antes de continuar.")

chatgpt_response = ask_chatgpt(globals().get("transcription"), SETTINGS["openai_model"])
print(chatgpt_response)

## 5. Gerar e ouvir a resposta

Esta etapa envia o texto da resposta ao serviço do Google utilizado pelo gTTS. O arquivo gerado é MP3. Se o navegador bloquear a reprodução automática, clique no botão de reprodução.

In [ ]:
response_audio = None
if not globals().get("_assistant_ready", False) or not globals().get("_preparation_complete", False) or "SETTINGS" not in globals():
    raise RuntimeError("Execute as células de configuração, preparação e funções, nesta ordem, antes de continuar.")

response_audio = synthesize_speech(globals().get("chatgpt_response"), SETTINGS["language"])
display(Audio(filename=response_audio, autoplay=True))

## Executar novamente e compartilhar

Para fazer outra pergunta, execute novamente as etapas **2 → 3 → 4 → 5**, nessa ordem. Se uma etapa falhar, corrija a causa e execute-a de novo antes de avançar. Uma falha não permite reaproveitar silenciosamente a transcrição ou resposta de uma pergunta anterior.

Antes de compartilhar: use **Editar → Limpar todas as saídas** e salve o notebook. Confira também os arquivos da sessão. A limpeza de saídas não remove commits antigos nem cópias já compartilhadas.

Consulte o README e a revisão técnica do repositório para testes locais e limitações. Os testes automatizados utilizam serviços simulados; não substituem a execução real desta sequência no Colab.